In [188]:
using LinearAlgebra, BenchmarkTools, PolynomialRoots, StaticArrays, DataStructures, Statistics, Roots

# Fully-split velocity Lagrangian PDMP (Technical WIP - not complete yet)
The main branch has the funcitonal code for the Split Lagrangian PDMP and Covariance Adaptive BPS. This part is under development.

## Mathemagical preliminaries
### A quick outline
Our goal shall be to define a set of exactly solvable problems to see that the fully-split velocity Lagrangian PDMP can be handled almost entirely analytically. First we establish what the rates and dynamics look like. Then we show that the rates are cubic in each velocity component, and that the dynamics are exactly solvable and rate integrals are possible to compute analytically. We shall compute and store the corresponding coefficients of the dynamics ODE and rates as tensors. Then we shall implement a program to compute the rates and update rules for the split-velocity part of the PDMP. This will enable us to merge this code with the previously written PDMP code to finally run this in parallell to CA-BPS or SL-PDMP.


### Splitting equations and general rate solution
Consider, for simplicity, a general split PDMP with each transitions $\alpha \to \beta$, i.e. only the split dynamic changes (Other events are entirely possible to consider in the same framework, but clutter up notation, so let us leave them out for now).  Let $\lambda(\alpha \to \beta)$ be the associated rates. To preserve a distribution $\mu$ the split flow is taken to satisfy

$
-\mu^{-1}\text{div}_\alpha(\mu \Phi^\alpha) = \sum_\beta  \lambda(\alpha \to \beta) - \lambda(\alpha \to \beta)
$

and denoting the divergence (sign?) on the LHS by $A_\alpha$, and $\lambda_{\alpha\beta} =\lambda(\alpha \to \beta) $ we get

$
A_\alpha = \sum_\beta \lambda_{\alpha\beta}-\lambda_{\beta\alpha}
$

Of course, if the original $\Phi$ satisfies $\text{div}(\mu \Phi)=0$ then so does the sum of split flows, as the sum of divergences $A_\alpha$ vanish by linearity of the divergence operator.

In trying to define the rates we have to solve a constraint on an anti-symmetric matrix. Clearly, adding any symmetric part to $\lambda_{\alpha\beta}$ is inconsequential for the constraint above. Effectively, under such a transformation, the altered flow ${\alpha \to \beta}$ is offset by an equivalent flow $\beta \to \alpha$, and in simulation we will get more or fewer events but the overall divergence is not altered. Since events are typically costly we generally will want $\lambda_{{\alpha\beta}}$ to have as small a symmetric part as possible. Alas, $\lambda_{{\alpha\beta}} \geq 0 $ so the symmetric part cannot be zero identically.

Rather than solving the above positivity and anti-symmetry constraints on $\lambda_{{\alpha\beta}}$ we can assume 

$
\lambda_{{\alpha\beta}} = [\rho_{\alpha\beta}]^+
$

where $\rho_{\alpha\beta}$ is some anti-symmetric matrix. Then 

$
\lambda_{\alpha\beta} - \lambda_{\beta\alpha}= [\rho_{\alpha\beta}]^+ - [\rho_{\beta\alpha}]^+ = [\rho_{\alpha\beta}]^+-[-\rho_{\alpha\beta}]^+ = \rho_{\alpha \beta}
$

and thus we have to solve

$
A_\alpha = \sum_\beta \rho_{\alpha\beta}
$

where the $A_\alpha$ are determined by $-\mu^{-1}\text{div}_\alpha(\mu \Phi_\alpha)$. A simple solution is given by

$
\rho_{\alpha \beta} = (A^\alpha-A^\beta)/n
$

where $n$ is the number of split states. This solution gives us a direct interpretation of the rates: it is possible to transition into a state $\beta$ _precisely_ when the divergence of flow associated to $\beta$ exceeds the divergence of the flow in the current state $\alpha$. The greater the difference in divergence, the more likely we are to transition into the given state.


Note that the $A_\alpha$ are arbitrary (but must satisfy $\sum_\alpha A_\alpha=0$), and so this is a valid solution and rates for any split. If for some reason we want to have a substantially increased rate we may adjust the rate matrix $\lambda_{\alpha\beta} \to \lambda_{\alpha\beta} + R_{\alpha\beta}$ where $R_{\alpha\beta}$ is any positive definite symmetric matrix. 

As we shall soon see there are several other solutions of interest.

### Preferential splitting rates
Suppose we are interested in a particular state, say, $\alpha = 0$, for which we want to have as small rates $\lambda_{0\beta}$ as possible. In other words, we would like 'stay' in the state $\alpha = 0$ for longer (or, at the very least, have fewer events $0 \to \beta$). Then, as we shall see, the above rates are not ideal. The rate $\lambda_{0 \to \text{any}} = \sum_\beta \lambda_{0\beta}$
becomes

$\lambda_{0 \to \text{any}} = \frac{1}{n+1}\sum_\beta [A_0-A_\beta]^+\geq \frac{1}{n+1}[\sum_\beta A_0-A_\beta]^+ = \frac{1}{n+1}[\sum_\beta A_0]^+ = [A_0]^+$

with equality if and only if $A_0 - A_\beta \geq 0$ for every $\beta$.

Had we only split into $0$ and taken the other splits $\beta= 1,2,\ldots, n $ as a single state $I$ the above 'recipe' would instead declare

$\lambda_{0 \to I} = \frac{1}{2}[A_0 - A_I]^+ =[A_0]^+$

since $A_0 + A_I = 0$. Clearly the former choice means that we transition into $I$ with a frequency that is *at least* as big as that of the latter choice, but generally larger.

Thus, if possible, we would like to use another rate for transitions $0 \to \beta$. Mercifully there are some obvious choices. Considering the equation

$A_0 = \sum_\beta \lambda_{0\beta}-\lambda_{\beta 0 }$

we can, as above, pick an anti-symmetric $\rho$-matrix and $\lambda_{0 \beta} = [\rho_{0\beta}]^+$ and $\lambda_{0 \beta} = [-\rho_{0\beta}]^+$  which leads to

$A_0 = \sum_\beta \rho_{0\beta}$

which we can solve by letting $\rho_{0\beta} = \frac{1}{n} A_0 = -\rho_{\beta 0}$ for $\beta \neq 0$. How should we interpret this choice for the rates? Now we only shift from $\alpha =0$ to _some_ (note that we effectively pick which uniformly at random) $\beta$ if the average divergence in $-A_\beta = A_0$ exceeds $0$, rather than if some individual divergence does so.

This has some consequence for the divergence equations for other splits. Let lower-case latin letters $a \neq 0$ index the $n$ splits other than $\alpha = 0$. We can in fact apply the same recipe as above again. To see this we have

$A_a = \sum_\alpha \lambda_{a\alpha}-\lambda_{\alpha a} =  (\lambda_{a0}-\lambda_{0a}) + \sum_b\lambda_{ab}-\lambda_{b a}  = \rho_{a0} + \sum_b\lambda_{ab}-\lambda_{b a} 
=-\frac{A_0}{n} + \sum_b\rho_{ab}$

Thus, picking $\rho_{ab} = \frac{1}{n}(A_a - A_b)$ we get

$ \sum_b\rho_{ab} = A_a - \frac{1}{n}\sum_b A_b = A_a  - \frac{1}{n}(-A_0 + \sum_{\beta} A_\beta ) = A_a + \frac{1}{n}A_0$

since $\sum_b A_b = -A_0 + \sum_\beta A_\beta = -A_0$ due to the vanishing overall divergence. Hence the $A_0$ terms in $a$-divergence equation cancel and the solution

$\rho_{ab} = \frac{1}{n}(A_a - A_b)$ 

is satisfactory.

### The Lagrangian split-velocity case
We now turn our attention to a splitting of the Lagrangian dynamics, so that we have one state corresponding to evolution of position, and one state for each evolution of a velocity component $v^i$. Thus, in sampling in $\mathbb{R}^n$ we shall have $n+1$ split states. In our sampling we shall treat position updates preferentially (
Shifting between velocities is *hopefully* inexpensive, but shifting between position and velocity updates is quite costly as it incurs a computation of third order derivatives.).

#### Notation, notation, my kingdom for better notation
We let the position flow correspond to the split state variable $\alpha = n+1$, so that the $I$:th velocity component can be chosen to be represented by $\alpha = I$, and so that matrix-enumerations match this (Matrices and arrays in Julia are, of course - as with any other sane language - indexed starting from 1.). We shall, in what comes, adopt a somewhat specialized notation. Namely, we let capital latin latters (typically $I,J,K,\ldots$) denote indices over a single dimension (the splitting dimension) which means we _**do not apply the Einstein summation convention**_ to these indices. By lower-case latin letters we refer to components that run across 'all dimensions other than the one currently evolving'. That is to say, if e.g. $\alpha = 3$ then a lower case letter $a$ tracks across all indices $1, 2, 4, 5, \ldots, n$. Greek letters refer always to the full set of indices $1,2,3,4,\ldots, n$. So if $\alpha = I$ we have

$\Gamma^\alpha_{\alpha \beta} = \Gamma^I_{I\beta} + \Gamma^{a}_{a\beta}$

but sometimes we encounter expressions like 

$\Gamma^\alpha_{J\beta}v^Jv^\beta = \Gamma^\alpha_{JI}v^Jv^I  + \Gamma^\alpha_{Ja}v^Jv^a$

where for $J\neq I$ the $a$ now does run over $J$, since $I$ is assumed to be evolving. Very tricky.

#### The equation of motion and divergences
The Lagrangian dynamics are described at length in the paper. Let us summarize some key features:

The flow in position, now associated to the $n+1$:th split state, is, componentwise:

$(\Phi_{n+1})^a = v^a$ 

and for the flow in the $I$:the velocity component we have (in a very slight abuse of notation we take $\Phi_I^I = \Phi^I$)

$\Phi^I = (-\eta - G^{-1}\nabla \phi)^I = -\Gamma^I_{\alpha\beta}v^\alpha v^\beta - G^{I\alpha}\partial_\alpha \phi = $

$= -\Gamma^I_{II} (v^I)^2 - 2\Gamma^I_{Ia}v^Iv^a-\Gamma^I_{ab}v^av^b-G^{I\alpha}\partial_\alpha \phi $

and since we do not (primarily) deal with BPS-type events we set $\phi = -\log \pi + \frac{1}{2}\log \det G$. However, since this is independent of $v$ it does not change the qualitative nature of the ODE defined by the flow along $\Phi_{I}$. We can express the flow in $v^J$ as a form of the _Ricatti equation_:

$du/dt = au^2 + b u + c $

for coefficients depending on $J$:

$ a_{;J} = -\Gamma^J_{JJ}$

$ b_{;J} = - 2\Gamma^J_{Ja}v^a$

$ c_{;J} = -G^{J\alpha}\partial_\alpha \phi-\Gamma^J_{ab}v^av^b$

The divergences for the velocities are straightforward to compute, but less pleasant to expand in our specialized indices. We shall need an expression for each divergence $A_I$ as a function of the evolving velocity component $u = v^J$. The divergences are cubic in the velocities so we shall expand $A_{I;J}(u) = A_{I;J}^{(0)} + A_{I;J}^{(1)}u^1+A_{I;J}^{(2)}u^2+A_{I;J}^{(3)}u^3 $ 

$A_I = -\mu^{-1}\text{div}_I(\mu \Phi_I) = (2\Gamma^I_{I\alpha} + \Phi^IG_{I\alpha})v^\alpha = 
2\Gamma^I_{I\alpha}v^\alpha - \Gamma^I_{\mu\nu}G_{I\alpha}v^\mu v^\nu v^\alpha - G^{I\beta} G_{I\alpha}(\partial_\beta \phi)v^\alpha $

and hence

$A_{I;J}^{(0)} = 2\Gamma^I_{Ia}v^a - \Gamma^I_{ab}G_{Ic}v^a v^b v^c - G^{I\beta} G_{Ia}(\partial_\beta \phi)v^a$

$A_{I;J}^{(1)} = 2\Gamma^I_{IJ} - 2\Gamma^I_{aJ}G_{Ib}v^a  v^b - \Gamma^I_{ab}G_{IJ}v^a v^b  - G^{I\beta} G_{IJ}(\partial_\beta \phi) $

$A_{I;J}^{(2)} = -\Gamma^I_{JJ} G_{Ia}v^a - 2\Gamma^I_{Ja}G_{IJ}v^a  $

$A_{I;J}^{(3)} = -\Gamma^I_{JJ}G_{IJ}$

The final divergence is

$A_{n+1; J} = -2\Gamma^\alpha_{\alpha \beta}v^\beta + \Gamma^\mu_{\alpha \beta} G_{\mu \nu} v^\alpha v^\beta v^\nu + (\partial_\alpha \phi)v^\alpha$

so (as might be inferred also from the above expression, and knowing that $A_{n+1} = \mu^{-1}\text{div}_x(\mu \Phi) = -\mu^{-1}\text{div}_v(\mu \Phi) = \sum_a A_a$)

$A_{n+1; J}^{(0)}  = -2\Gamma^\alpha_{\alpha a}v^a + \Gamma^\mu_{ab} G_{\mu c} v^a v^b v^c + (\partial_a \phi)v^a$

$A_{n+1; J}^{(1)}  = -2\Gamma^\alpha_{\alpha J} + 2\Gamma^\mu_{J a} G_{\mu b} v^a v^b + \Gamma^\mu_{ab } G_{\mu J} v^a v^b + (\partial_J \phi)$

$A_{n+1; J}^{(2)}  =  2\Gamma^\mu_{J a} G_{\mu J} v^a + \Gamma^\mu_{JJ}G_{\mu a}$

$A_{n+1; J}^{(3)}  =  \Gamma^\mu_{J J} G_{\mu J} $

Of course, to compute $A_{n+1}$ we should certainly use the form of the $A_a$ above. Since, at all times, we are only interested in a single $J$ at a time, and the relationships between $A_{I;J}$ and $A_{I;K}$ are somewhat complicated, we do not benefit greatly from computing the full $A_{I;J}^{(n)}$.

## Implementing the flow, divergences, rates etc

In [189]:
#Computing the object A^{(n)}_{I;J}:
function compute_divergences!(A, reduced_v, dim, J,  Γ, G, G_inv, ∇φ, vel)
    reduced_v .= vel
    reduced_v[J] = 0.0
    
    @views A[dim+1, :] .= 0.0

    @inbounds for I in 1:dim
        @views A[I, 4] = -Γ[I, J, J] * G[I, J]
        
        A[dim+1, 4] -= A[I, 4]

        @views A[I, 3] = -Γ[I, J, J] * dot(G[I, :], reduced_v) - 2 * G[I, J] * dot(Γ[I, J, :], reduced_v)

        A[dim+1, 3] -= A[I, 3]

        @views A[I, 2] = 2*Γ[I, I, J] - 2*dot(Γ[I, J, :], reduced_v)*dot(G[I, :], reduced_v) - G[I, J] * dot(reduced_v, Γ[I, :, :], reduced_v) - G[I, J] * dot(G_inv[I, :], ∇φ)

        A[dim+1, 2] -= A[I, 2]

        @views A[I, 1] = 2*dot(Γ[I, I, :], reduced_v) - dot(reduced_v, Γ[I, :, :], reduced_v)*dot(G[I, :], reduced_v) - dot(G[I, :], reduced_v) * dot(G_inv[I, :], ∇φ)

        A[dim+1, 1] -= A[I, 1]
    end

    return A
end        

compute_divergences! (generic function with 1 method)

As a rough order of magnitude estimate this should take something like $\sim 10 \mu s$ to compute for a 20-dimensional space.

#### $J\to I$ and $J\to 0$ rate under $J$-flow from the divergences
We have $\rho_{JI} = (A_{J;J}-A_{I;J})/n $ and $\rho_{J , n+1} = -A_{n+1;J}/n$ so given the above we can compute the rates with ease. Since we can solve each flow (almost) exactly we shall not have need of the explicit rates, but rather the $\rho_{JI}^{(n)}$

In [190]:
function compute_rho!(ρJ, J, A, dim)
    @inbounds for I in 1:dim
        if I ≠ J
            @views ρJ[I,:] .= A[J,:] .- A[I,:]
        else
            @views ρJ[J,:] .= 0.0
        end
    end

    @views ρJ[n+1, :] .= (-A[n+1,:] ./ dim) 

    return ρJ
end

compute_rho! (generic function with 1 method)

### Ricatti equation and the velocity flow
The fully split velocity satisfies the Ricatti equation

$du/dt = a u^2 + b u + c$

for some $a,b,c$. This admits the generic (but not general!) solution

$u(t) = (\kappa \tan(\kappa(t+t_0))-(b/2))/a$

where $\kappa = \sqrt{4ac-b^2}/2$. There are, however, numerous edge cases that must be handled when e.g. $a=0$ or $\kappa = 0$. These cause a substantial headache as they must be integrated (for exact rate integrals) to the $n$:th power for $n=1,2,3$.

If $a \neq 0$ then we can transform into a reduced form by letting $y = u a$ and we get a reduced equation $y' = y^2 + b y + ca$ where $y_0 = a u_0$.

Before we move on, let us implement a method to compute $a,b,c$

In [191]:
function compute_velocity_parameters!(vred, Γ, Ginv, ∇φ, J, v)
    vred .= v
    vred[J] = 0.0

    a = -Γ[J,J,J] 
    @views b = -2*dot(Γ[J, J,:],vred)

    @views c = - dot(Ginv[J,:], ∇φ) - dot(vred, Γ[J,:,:], vred)

    return (a,b,c)
end

compute_velocity_parameters! (generic function with 1 method)

In [192]:
dim = 5
vred = randn(dim)
Γ = randn(dim, dim, dim) #wrong symmetries, but whatever
Ginv = randn(dim, dim)#wrong symmetries, but whatever
∇φ = randn(dim)
v = randn(dim)
J = rand(1:dim)

5

In [193]:
@benchmark compute_velocity_parameters!($vred, $Γ, $Ginv, $∇φ, $J, $v)

BenchmarkTools.Trial: 10000 samples with 964 evaluations per sample.
 Range (min … max):  78.112 ns … 331.432 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     82.780 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   98.941 ns ±  31.250 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▃▇█▂▂▄▃  ▁▂ ▁▁▄▅▁ ▁▁   ▁                        ▁            ▁
  ███████████▇████████████▆▆▆█▇▅▄▅▆▅▆▅▆▅▅▅▄▄▇▇▇███████████▇▆▇▇ █
  78.1 ns       Histogram: log(frequency) by time       208 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

### The 6 solutions of the Ricatti equation
We shall have to include Ricatti-equation tests, comparing $u'(t)$ with $au^2+bu+c$ for all interesting variable combinations. Above we noted that we essentially had 6 distinct solutions (which I've enumerated in the ~Pythonic~ way for who knows what reason):

0) Stationary solutions for which $u = u_0$
1) (Non-stationary) Solutions for $a = b= 0, c\neq 0$, leading to $u(t) = c t + u_0$
2) (Non-stationary) Solutions for $a = 0, b\neq 0, c\neq 0$, leading to $u(t) = (u_0 + p)e^{bt} - p$ where $p = c/b$
3) (Non-stationary) Solutions with $a\neq 0$ but $\kappa^2 = 4ac -b^2 =0$ (and so $c = b^2/(4a)$) for which 

    $u(t) =  \frac{1}{a}(\frac{-b}{2} + \frac{l_0}{1 - l_0t})$ where $l_0 =  (\frac{b}{2} + au_0)$.
    
4) (Non-stationary) Solutions with $a\neq 0$ and $\kappa^2 = ac -(b^2/4) > 0$, $\kappa = \sqrt{|ac -(b^2/4)|}$. Then 

    $u(t) = a^{-1}(\kappa  \tan(s_0 + (\kappa t)) - q) $ where $q =  b/2$ and $s_0$ is such that $u(0)=u_0$.

5) (Non-stationary) Solutions with $a\neq 0$ and $\kappa^2 = ac -(b^2/4) < 0$ and $\kappa = \sqrt{|ac -(b^2/4)|}$. Then 

    $u(t) = a^{-1}(-\frac{b}{2} + \kappa(-1 + 2(1-\tilde\omega_0e^{2\kappa t})^{-1})) $ and $\tilde\omega_0 = 1 - \frac{2\kappa}{u_0 a + b/2 + \kappa}$. Note that $u_0 \neq -\frac{b}{2} +  \kappa$ since we assume the solution isn't stationary.

These solutions have one thing in common (the first case could arguably be excluded): The sign of $du/dt$ only depends on some constant in each case - thus it is simple to see what range of future $u$ we have, given $u_0$ and this determining parameter. We have

1) $c$ since $u'(t) = c$ 
2) $b$ since $u'(t) = (pb+u_0b) e^{bt} = (c+u_0b)e^{bt}$ which has the sign of $c + u_0b$
3) $a$ since $u'(t) = a^{-1}l_0^2(1-l_0t)^{-2}$ which has the sign of $a$
4) $a$ since $u'(t) = a^{-1} \kappa^2 (1+ \tan^2(s_0+\kappa t))$ which has the sign of $a$
5) $a$ since $u'(t) = 2a^{-1}\kappa^2 \omega_0 e^{2\kappa t} (1-\omega_0\exp(2\kappa t))^{-2}$ which has the sign of $a\omega_0$

In [194]:
#There are a lot of exact float checks here. This has to be managed/studied to see where issues arise.
function reduced_ricatti_solution(t, β, γ; y0 = 0.0)
    if isapprox(4*γ, β^2)
        l0 = y0+(β/2)
        return (l0*inv(1-(l0*t)))-(β/2)  #Type 3 velocity
    end 
    κ2 = γ - ((β^2)/4)
    k = sqrt(abs(κ2))
    δ = β/2 
    if κ2 > 0
        s0 = atan((y0 + δ)/k)# t0 = s0/k
        return (k * tan(s0 + (k*t))) - δ #Type 4 velocity
    else 
        ω_0 = 1 - (2*k/(y0 + δ + k)) 
        return -δ + (k*(-1 + (2*inv(1 - ω_0*exp(2*k*t))))) #Type 5 velocity
    end
end

function ricatti_solution(t,a,b,c; u0 = 0.0, verbose = false)
    if isapprox(a*(u0^2) + (b*u0) , -c)
        verbose ? (@warn "Stationary velocity, rates are constant") : nothing
        return u0 #Type 0 velocity
    end
    if a ≈ 0 
        if b ≈ 0 
            #Cant have c=0 in this case as that would trigger the above constraint. Of course, in really dicey accuracy questions this might be relevant.
            if c ≈ 0
                verbose ? (@warn "Stationary velocity, rates are constant") : nothing
                return u0 #Type 0 velocity
            end
            return c*t + u0 #Type 1 velocity
        end
        q = c/b
        return (u0 + q)*exp(b*t) - q #Type 2 velocity
    end
    return reduced_ricatti_solution(t, b, c*a; y0 = a*u0)/a
end

ricatti_solution (generic function with 1 method)

### Tests

In [195]:
function rel_failure(v_approx, v)
    if v ≈ v_approx
        return 0.0
    end
    if iszero(v)
        if iszero(v_approx)
            return 0.0
        end
        return 1.0
    end
    return abs(v_approx - v)/(abs(v_approx) + abs(v))
end

rel_failure (generic function with 1 method)

#### Test of solution 0

In [196]:
function test_0(;t_range = 0.0:0.001: 3.0)
    a = (10^rand())*randn()
    b = (10^rand())*randn()
    c = (b^2)/4
    u0 = (10^rand())*randn()
    c = - (a*(u0^2)) - (b*u0)
    u_expected(t) = u0
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = 0.0
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    failure_in_value = 0.0
    failure_in_der = 0.0
    n = 0
    for t in t_range
        failure_in_value += rel_failure(u(t), u_expected(t))
        failure_in_der += rel_failure(du(t), du_expected(t))
        n+=1
    end
    if failure_in_der ≠ 0 || failure_in_value ≠ 0
        println("Total relative failure from expected solution: $failure_in_value \n Total relative failure in derivative $failure_in_der \n (over $n samples)")
    end
    nothing
end

test_0 (generic function with 1 method)

In [197]:
for i = 1:100
    test_0()
end

#### Test of solution 1

In [198]:
function test_1(;t_range = 0.0:0.001: 3.0)
    a = 0.0
    b = 0.0
    c = (10^rand())*randn()
    while c ≈ 0
        c = (10^rand())*randn()
    end
    u0 = (10^rand())*randn()
    u_expected(t) = c*t + u0
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = c
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    failure_in_value = 0.0
    failure_in_der = 0.0
    n = 0
    for t in t_range
        failure_in_value += rel_failure(u(t), u_expected(t))
        failure_in_der += rel_failure(du(t), du_expected(t))
        n+=1
    end
    if failure_in_der ≠ 0 || failure_in_value ≠ 0
        println("Total relative failure from expected solution: $failure_in_value \n Total relative failure in derivative $failure_in_der \n (over $n samples)")
    end
    nothing
end

test_1 (generic function with 1 method)

In [199]:
for i = 1:100
    test_1()
end

#### Test of solution 2

In [200]:
function test_2(;t_range = 0.0:0.0001: 3.0, threshold = 10^(-16))
    a = 0.0
    b = (10^rand())*randn()

    while b ≈ 0
        b = (10^rand())*randn()
    end

    c = (10^rand())*randn()
    while c ≈ 0
        c = (10^rand())*randn()
    end

    u0 = (10^rand())*randn()
    while (b*u0) + c ≈ 0 || c ≈ 0
        c = (10^rand())*randn()
    end

    p = c/b
    u_expected(t) = (u0 + p)*exp(b*t) - p
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = (u0 + p)*b*exp(b*t)
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    
    n=0
    rel_val_failure = Float64[]
    rel_der_failure = Float64[]
    for t in t_range
        push!(rel_val_failure, rel_failure(u(t), u_expected(t)))
        push!(rel_der_failure, rel_failure(du(t), du_expected(t)))
        n+=1
    end
    failure_in_value = sum(rel_val_failure)
    failure_in_der = sum(rel_der_failure)
    
    mv = maximum(rel_val_failure)
    md = maximum(rel_der_failure)
   
    
    if mv > threshold || md > threshold 
        if failure_in_value ≠ 0
            println("Failure in val! Total relative failure from expected solution: $failure_in_value")
            ind = findfirst(x-> x==mv, rel_val_failure) #I know, inefficient
            t= t_range[ind]
            μval = mean(rel_val_failure)
            σval = std(rel_val_failure)
            println("Failure in val \n Maximal failure: $mv at t = $t\n Mean value failure: $μval \n Standard deviation: $σval \n Value: $((u(t), u_expected(t)))")
        end

        if failure_in_der ≠ 0
            println("Failure in der! Total relative failure from expected solution: $failure_in_der")
            ind = findfirst(x-> x==md, rel_der_failure) #I know, inefficient
            t= t_range[ind]
            μder = mean(rel_der_failure)
            σder = std(rel_der_failure)
            println("Maximal failure: $md at t = $t\n Mean der failure: $μder \n Standard deviation: $σder \n Value: $((du(t), du_expected(t)))")
        end
        println("Coefficients: (a,b,c, u0) = $((a,b,c,u0))")
    end
    nothing
end

test_2 (generic function with 1 method)

In [201]:
for i = 1:100
    test_2()
end

Failure in der! Total relative failure from expected solution: 10.604583297008189
Maximal failure: 0.04134273693220945 at t = 2.9988
 Mean der failure: 0.0003534743274226922 
 Standard deviation: 0.0019280914226636335 
 Value: (1.3322676295501878e-15, 1.4471774982879096e-15)
Coefficients: (a,b,c, u0) = (0.0, -12.655895856223847, -0.5166396297127216, -3.5145071657341664)
Failure in der! Total relative failure from expected solution: 165.50353924746744
Maximal failure: 0.3469072795686611 at t = 2.9425
 Mean der failure: 0.005516600754890418 
 Standard deviation: 0.029757162113280104 
 Value: (-2.220446049250313e-16, -4.579342035307746e-16)
Coefficients: (a,b,c, u0) = (0.0, -13.211584732028268, 1.3200610567337634, 2.748931841989754)
Failure in der! Total relative failure from expected solution: 58.15200585998314
Maximal failure: 0.12520709520063156 at t = 2.9645
 Mean der failure: 0.0019383355841466332 
 Standard deviation: 0.009767176525766962 
 Value: (6.661338147750939e-16, 8.568182144

Here the derivative fails in many tests. Fundamentally what appears to be the issue is that large or small exponentials are added to a float of some *reasonable* size. It is generally a difficult issue that we shall not attempt to overcome.

Up to float-issues, the test appears to pass - and since our only issues are with small differences of the derivative we feel certain about the validity of the solution. Note that the derivatives are not explicitly used in the computations ahead - they are present here as a sanity check on our "solutions" as actual solutions of the ODE.

#### Test of solution 3

$u(t) =  \frac{1}{a}(\frac{-b}{2} + \frac{l_0}{1 - l_0t})$ where $l_0 =  (\frac{b}{2} + au_0)$

In [202]:
function test_3(;t_range = 0.0:0.0001: 3.0, threshold = 10^(-15))
    a = (10^rand())*randn()
    while a ≈ 0
        a = (10^rand())*randn()
    end
    b = (10^rand())*randn()
    c = (b^2)/(4*a)

    u0 = (10^rand())*randn()
    while a*(u0^2) + b*(u0) + c ≈ 0 
        u0 = (10^rand())*randn()
    end

    l_0 = (a*u0) + (b/2)
    u_expected(t) = ((-b/2) + (l_0*inv(1 - (l_0*t))))*inv(a)
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = l_0^2*((inv(1 - (l_0*t)))^2)/a
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    
    n=0
    rel_val_failure = Float64[]
    rel_der_failure = Float64[]
    for t in t_range
        push!(rel_val_failure, rel_failure(u(t), u_expected(t)))
        push!(rel_der_failure, rel_failure(du(t), du_expected(t)))
        n+=1
    end
    failure_in_value = sum(rel_val_failure)
    failure_in_der = sum(rel_der_failure)
    
    mv = maximum(rel_val_failure)
    md = maximum(rel_der_failure)
   
    if mv > threshold || md > threshold 
        if failure_in_value ≠ 0
            println("Failure in val! Total relative failure from expected solution: $failure_in_value")
            ind = findfirst(x-> x==mv, rel_val_failure) #I know, inefficient
            t= t_range[ind]
            μval = mean(rel_val_failure)
            σval = std(rel_val_failure)
            println("Failure in val \n Maximal failure: $mv at t = $t\n Mean value failure: $μval \n Standard deviation: $σval \n Value: $((u(t), u_expected(t)))")
        end

        if failure_in_der ≠ 0
            println("Failure in der! Total relative failure from expected solution: $failure_in_der")
            ind = findfirst(x-> x==md, rel_der_failure) #I know, inefficient
            t= t_range[ind]
            μder = mean(rel_der_failure)
            σder = std(rel_der_failure)
            println("Maximal failure: $md at t = $t\n Mean der failure: $μder \n Standard deviation: $σder \n Value: $((du(t), du_expected(t)))")
        end
        println("Coefficients: (a,b,c, u0) = $((a,b,c,u0))")
    end
    nothing
end

test_3 (generic function with 1 method)

In [203]:
for i=1:100
    test_3()
end

#### Test of solution 4
$u(t) = a^{-1}(\kappa  \tan(s_0 + (\kappa t)) - q) $ where $q =  b/2$ and $s_0$ is such that $u(0)=u_0$, $\kappa^2 = ac - b^2/2$ 

In [204]:
function test_4(;t_range = 0.0:0.0001: 3.0, threshold = 10^(-15))
    a = (10^rand())*randn()
    while a ≈ 0
        a = (10^rand())*randn()
    end
    b = (10^rand())*randn()
    
    κ2 = (10^rand())*(randn()^2) 
    while κ2 ≈ 0
        κ2 = (10^rand())*(randn()^2)
    end

    c = (κ2 + ((b^2)/4))/a

    u0 = (10^rand())*randn()
    while a*(u0^2) + b*(u0) + c ≈ 0 
        u0 = (10^rand())*randn()
    end

    κ = sqrt(κ2)
    s0 = atan((a*u0 + (b/2))/κ)


    u_expected(t) = inv(a)*(-(b/2) + κ*tan(s0 + (κ*t)))
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = (κ^2)*inv(a)*(1 + (tan(s0 + (κ*t))^2))
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    
    n=0
    rel_val_failure = Float64[]
    rel_der_failure = Float64[]
    for t in t_range
        push!(rel_val_failure, rel_failure(u(t), u_expected(t)))
        push!(rel_der_failure, rel_failure(du(t), du_expected(t)))
        n+=1
    end
    failure_in_value = sum(rel_val_failure)
    failure_in_der = sum(rel_der_failure)
    
    mv = maximum(rel_val_failure)
    md = maximum(rel_der_failure)
   
    if mv > threshold || md > threshold 
        if failure_in_value ≠ 0
            println("Failure in val! Total relative failure from expected solution: $failure_in_value")
            ind = findfirst(x-> x==mv, rel_val_failure) #I know, inefficient
            t= t_range[ind]
            μval = mean(rel_val_failure)
            σval = std(rel_val_failure)
            println("Failure in val \n Maximal failure: $mv at t = $t\n Mean value failure: $μval \n Standard deviation: $σval \n Value: $((u(t), u_expected(t)))")
        end

        if failure_in_der ≠ 0
            println("Failure in der! Total relative failure from expected solution: $failure_in_der")
            ind = findfirst(x-> x==md, rel_der_failure) #I know, inefficient
            t= t_range[ind]
            μder = mean(rel_der_failure)
            σder = std(rel_der_failure)
            println("Maximal failure: $md at t = $t\n Mean der failure: $μder \n Standard deviation: $σder \n Value: $((du(t), du_expected(t)))")
        end
        println("Coefficients: (a,b,c, u0) = $((a,b,c,u0))")
    end
    nothing
end

test_4 (generic function with 1 method)

In [205]:
for i = 1:100
    test_4()
end

This appears generally stable.

#### Test of solution 5


$u(t) = a^{-1}(-\frac{b}{2} + \tilde\kappa(-1 + 2(1-\tilde\omega_0e^{2\tilde\kappa t})^{-1})) $ where $\tilde\kappa = \sqrt{b^2/4 - ac}$ and $\tilde\omega_0 = 1 - \frac{2\tilde\kappa}{u_0 a + b/2 + \tilde\kappa}$. Note that $u_0 \neq -\frac{b}{2} - \tilde \kappa$ since we assume the solution isn't stationary.

In [206]:
function test_5(;t_range = 0.0:0.0001: 3.0, threshold = 10^(-15))
    a = (10^rand())*randn()
    while a ≈ 0
        a = (10^rand())*randn()
    end
    b = (10^rand())*randn()
    
    κ2 = -(10^rand())*(randn()^2) 
    while κ2 ≈ 0
        κ2 = -(10^rand())*(randn()^2)
    end

    c = (κ2 + ((b^2)/4))/a

    u0 = (10^rand())*randn()
    while a*(u0^2) + b*(u0) + c ≈ 0 
        u0 = (10^rand())*randn()
    end

    κ = sqrt(abs(κ2))
    ω0 = 1- (2*κ*inv((u0*a) + (b/2) + κ))


    u_expected(t) = inv(a)*(-(b/2) + κ*(-1 + (2*inv(1-ω0*exp(2*κ*t)))))
    u(t) = ricatti_solution(t, a,b,c, u0 = u0, verbose = false)
    du_expected(t) = ((2*κ)^2)*inv(a)*ω0*((inv(1 - ω0*exp(2*κ*t)))^2)*exp(2*κ*t)
    du(t) = (a*(u(t)^2)) + (b * u(t)) + c
    
    n=0
    rel_val_failure = Float64[]
    rel_der_failure = Float64[]
    for t in t_range
        push!(rel_val_failure, rel_failure(u(t), u_expected(t)))
        push!(rel_der_failure, rel_failure(du(t), du_expected(t)))
        n+=1
    end
    failure_in_value = sum(rel_val_failure)
    failure_in_der = sum(rel_der_failure)
    
    mv = maximum(rel_val_failure)
    md = maximum(rel_der_failure)
   
    if mv > threshold || md > threshold 
        if failure_in_value ≠ 0
            println("Failure in val! Total relative failure from expected solution: $failure_in_value")
            ind = findfirst(x-> x==mv, rel_val_failure) #I know, inefficient
            t= t_range[ind]
            μval = mean(rel_val_failure)
            σval = std(rel_val_failure)
            println("Failure in val \n Maximal failure: $mv at t = $t\n Mean value failure: $μval \n Standard deviation: $σval \n Value: $((u(t), u_expected(t)))")
        end

        if failure_in_der ≠ 0
            println("Failure in der! Total relative failure from expected solution: $failure_in_der")
            ind = findfirst(x-> x==md, rel_der_failure) #I know, inefficient
            t= t_range[ind]
            μder = mean(rel_der_failure)
            σder = std(rel_der_failure)
            println("Maximal failure: $md at t = $t\n Mean der failure: $μder \n Standard deviation: $σder \n Value: $((du(t), du_expected(t)))")
        end
        println("Coefficients: (a,b,c, u0) = $((a,b,c,u0))")
    end
    nothing
end

test_5 (generic function with 1 method)

In [207]:
for i=1:100
    test_5()
end

Failure in der! Total relative failure from expected solution: 794.771357653075
Maximal failure: 1.0 at t = 2.9636
 Mean der failure: 0.02649149553858455 
 Standard deviation: 0.12562861792667235 
 Value: (0.0, 1.59453018462438e-14)
Coefficients: (a,b,c, u0) = (-0.2932032869280818, -0.1201992927718496, 143.7726812517349, -4.156182682872872)
Failure in der! Total relative failure from expected solution: 10.692357571895858
Maximal failure: 0.027581859473598723 at t = 2.9829
 Mean der failure: 0.0003564000390618932 
 Standard deviation: 0.0016547228793798064 
 Value: (8.348877145181177e-14, 7.900684032276884e-14)
Coefficients: (a,b,c, u0) = (3.272700796412478, 0.4285129091092499, -9.596102038569219, -9.65172369050065)
Failure in der! Total relative failure from expected solution: 0.3022546739369378
Maximal failure: 0.0008078376394269841 at t = 2.9984
 Mean der failure: 1.0074819970565573e-5 
 Standard deviation: 4.4456686547584716e-5 
 Value: (-4.137135078963183e-12, -4.130456207534552e-1

### A more efficient form of the above
While the above reassures us of the correctness of our solutions we'd like a more efficient form of the various forms of $u(t)$ and its inverses $t(u)$.

In [208]:
struct VelocityFunctions{T,F,G}
    velocity::T
    inverse::F
    integral::G #Later we shall need to be able to compute the integral
end

function evaluate(V::VelocityFunctions, t, param)
    return V.velocity(t, param...)
end

function evaluate_inverse(V::VelocityFunctions, u, param)
    return V.inverse(u, param...)
end

function evaluate_integral(V::VelocityFunctions, t, param)
    return V.integral(t, param...)
end

evaluate_integral (generic function with 1 method)

In [209]:
#We want a method for the sign that returns an integer no matter what (unlike the default sign function in Julia, which returns the sign with corresponding type)
function sgn(x)
    if x < 0
        return -1
    elseif x > 0
        return 1
    end
    return 0
end

sgn (generic function with 1 method)

To avoid allocations we define the type functions, and their inverses.¨

0. $u(t) = u_0$
1. $u(t) = c t + u_0$
2. $u(t) = (u_0 + p)e^{bt} - p$ 
3. $u(t) =  \frac{1}{a}(\delta + \frac{l_0}{1 - l_0t})$ 
4. $u(t) = a^{-1}(\kappa  \tan(s_0 + (\kappa t)) - \delta) $
5. $u(t) = a^{-1}(\delta + \kappa(-1 + 2(1-\tilde\omega_0e^{2\kappa t})^{-1})) $

The inverses are:

1. $t(u) = (u-u_0)/c$
2. $t(u) = \frac{1}{b}\ln(\frac{u+p}{u_0+p}) $
3. $t(u) = \frac{1}{l_0}-\frac{1}{au+\delta } $
4. $t(u) = \frac{1}{\kappa}(-s_0 + \arctan\frac{au+\delta}{\kappa})$
5. $t(u) = \frac{1}{2\kappa}\ln\frac{au+\delta -\kappa}{\omega_0(au+\delta +\kappa)}$

##### Type 0

In [210]:
#Parameters: (u0)
function type_0(t, u0)
    return u0
end

#type_0 obviously has no inverse
type0 = VelocityFunctions(type_0, :None, :None)

VelocityFunctions{typeof(type_0), Symbol, Symbol}(Main.type_0, :None, :None)

##### Type 1

In [211]:
#Parameter tuple: c, u0, inv_c
function type_1(t, c, u0, inv_c)
    return c*t + u0
end

function type_1_inv(u, c, u0, inv_c)
    return (u-u0)*inv_c
end

type1 = VelocityFunctions(type_1, type_1_inv, G_type_1)

VelocityFunctions{typeof(type_1), typeof(type_1_inv), typeof(G_type_1)}(Main.type_1, Main.type_1_inv, Main.G_type_1)

##### Type 2

In [212]:
#Param b, q, m
function type_2(t, b, q, m)
    return m*exp(b*t)-q
end

function type_2_inv(u, b,q, m)
    return log((u+q)/m)/b
end

type2 = VelocityFunctions(type_2, type_2_inv, G_type_2)

VelocityFunctions{typeof(type_2), typeof(type_2_inv), typeof(G_type_2)}(Main.type_2, Main.type_2_inv, Main.G_type_2)

##### Type 3

In [213]:
#Parameters (a_inv, l0_inv, δ, a)
function type_3(t, a_inv, l0_inv, δ, a)
    #a_inv = 1/a
    #l0_inv = 1/(a*u_0 + δ)
    return a_inv*((1/(l0_inv-t)) - δ)
end

function type_3_inv(u, a_inv, l0_inv, δ, a)
    return l0_inv - (1/(a*u + δ))
end

type3 = VelocityFunctions(type_3, type_3_inv, G_type_3)

VelocityFunctions{typeof(type_3), typeof(type_3_inv), typeof(G_type_3)}(Main.type_3, Main.type_3_inv, Main.G_type_3)

##### Type 4

In [ ]:
#Parameters (a_inv, k, s0, δ, k_inv, a)
function type_4(t, a_inv, k, s0, δ, k_inv, a)
    return a_inv*((k * tan(s0 + (k*t))) - δ)
end

function type_4_inv(u, a_inv, k, s0, δ, k_inv, a) #not a proper inverse! Inverts Tan - but |s0+kt| may exceed π/2
    return k_inv*(-s0 + atan(k_inv*((a*u)+δ)))
end

type4 = VelocityFunctions(type_4, type_4_inv, G_type_4)

VelocityFunctions{typeof(type_4), typeof(type_4_inv), typeof(G_type_4)}(Main.type_4, Main.type_4_inv, Main.G_type_4)

##### Type 5

In [215]:
#Parameters: (a_inv, δk, dk, ω0, a)
function type_5(t, a_inv, δk, dk, ω0, a)
    return a_inv*(-δk + (dk/(1 - ω0*exp(dk*t))))
end

function type_5_inv(u, a_inv, δk, dk, ω0, a)
    return log((1-(dk/((a*u) + δk)))/ω0)/dk
end

type5 = VelocityFunctions(type_5, type_5_inv, G_type_5)

VelocityFunctions{typeof(type_5), typeof(type_5_inv), typeof(G_type_5)}(Main.type_5, Main.type_5_inv, Main.G_type_5)

In [248]:
function generate_test_param()
    #Parameter tuple: c, u0, inv_c
    param1 = rand(3)
    param1[end] = 1/param1[1]

    #Parameters (b, q, m)
    param2 = rand(3)
    param2[end] = 1/param2[1]

    #Parameters (a_inv, l0_inv, δ, a)
    param3 = rand(4)
    param3[end] = 1/param3[1]
    
    #(a_inv, k, s0, δ, k_inv, a)
    param4 = rand(6)
    param4[2] *= 0.05
    param4[3] *= 0.3
    param4[end] = 1/param4[1]
    param4[end-1] = 1/param4[2]

    #Parameters: (a_inv, δk, dk, ω0, a)
    param5 = rand(5)
    δk = rand()
    u0 = rand()
    param5[4] = -inv(1-(param5[3]/(param5[5]*0.5 + param5[2])))
    param5[end] = 1/param5[1]
    return param1,param2, param3, param4, param5
end

generate_test_param (generic function with 1 method)

In [246]:
function test_ricatti_solution(;times = rand(10))
    param1, param2, param3, param4, param5 = generate_test_param()
    for t in times
        t1 = type_1_inv(type_1(t,param1...), param1...)
        if !(t1 ≈ t)
            @show t1, t
        end
        
        t2 = type_2_inv(type_2(t,param2...), param2...)
        if ! (t2 ≈ t)
            @show t2, t
        end
        
        t3 = type_3_inv(type_3(t,param3...), param3...)
        if ! (t3 ≈ t)
            @show t3, t
        end
        
        t4 = type_4_inv(type_4(t,param4...), param4...)
        if ! (t4 ≈ t)
            @show t4, t
            @show param4
        end
        
        t5 = type_5_inv(type_5(t,param5...), param5...)
        if ! (t5 ≈ t)
            @show t5, t
        end
    end
end

test_ricatti_solution (generic function with 1 method)

In [ ]:
#May fail in case 4 when parameters are such that we take arctan(tan(u)) for u>π/2
for j=1:100
    test_ricatti_solution()
end

### The function that we actually call to figure out velocities
To figure out the velocity we feed in the parameters $a,b,c,u_0$ and get our dynamics and (dynamic specific) parameters that we use to compute rates and partitions.

In [219]:
function direction_velocity_and_type(a,b,c,u0)
    if isapprox(a*(u0^2) + (b*u0) , -c)
        @warn "Stationary velocity, rates are constant"
        return (0,  (u0,), type0)
    end
    if a ≈ 0 
        if b ≈ 0 
            if c ≈ 0 #Techincally shouldn't happen, but floats are floats
                @warn "Stationary velocity, rates are constant"
                return (0,  (u0,), type0)
            end
            return return (sgn(c), (c, u0), type1)
        end
        q = c/b
        return (sgn(u0*b + c), (b, q, u0+b), type2)
    end
    #β = b
    #γ = c*a
    #y0 = a*u0
    δ = b/2
    l0 = a*u0 + δ
    X1 = 4*c*a
    X2 = b^2
    a_inv = inv(a)
    if isapprox(X1, X2)
        l0_inv = inv(l0)         
        return (sgn(a), (a_inv, l0_inv, δ, a), type3)
    end 
    κ2 = X1-X2
    k = sqrt(abs(κ2))
    if κ2 > 0
        s0 = atan(l0/k)# t0 = s0/k
        return (sgn(a), (a_inv, k, s0, δ, inv(k), a), type4)

    elseif κ2 <0 #should be an 'else', but we'd rather get errors than bugs
        ω_0 = 1 - (2*k/(l0 + k)) 
        dk = 2*k 
        return (sgn(ω_0*a), (a_inv, k, dk, ω_0, δ, inv(dk), inv(ω_0), a), type5)
    end
    error("Comparison failure.")
end

direction_velocity_and_type (generic function with 1 method)

## Exact methods for rate integrals
### Exact methods to partition time space
The rates are given by some expression

$\lambda^{IJ} = [\rho^{IJ}]^+ = [A^I - A^J]^+/n$

and the total rate in state $I$ is simply the sum $\lambda^I = \sum_J\lambda^{IJ}$. As discussed above the $A^I$ are cubic in $u(t)$. We can explicitly integrate each $\lambda^{IJ}$ *if* we know that it is positive. Thus, for a given $I$, we solve $\rho^{IJ}(u) = 0$ for all $J$ and order the individual solutions $u_1, u_2, \ldots$ such that $u_i < u_{i+1}$ if $du/dt > 0$, and $u_i> u_{i+1}$ else (Note that $du/dt$ is actually identically positive or negative (or zero!) for all $t$). This partitions the time space $U$ into sets over which the signs of all $\rho^{IJ}$ are constant and over these the sum $\sum_J \lambda^{IJ}$ can thus be computed.

#### Finding roots of cubics

In [220]:
pol = [9,3,-7,1]
@benchmark roots($pol)

BenchmarkTools.Trial: 10000 samples with 195 evaluations per sample.
 Range (min … max):  479.487 ns … 204.434 μs  ┊ GC (min … max): 0.00% … 99.50%
 Time  (median):     582.564 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   672.502 ns ±   2.550 μs  ┊ GC (mean ± σ):  8.51% ±  2.93%

   █▅                                                            
  ▅██▆▃▂▃▄▅▄▃▅█▆▅▃▃▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  479 ns           Histogram: frequency by time         1.27 μs <

 Memory estimate: 496 bytes, allocs estimate: 8.

In [221]:
"""
    cubic_real_roots(b, c, d; verbose=false)::Tuple

Finds the real roots of the cubic polynomial x^3 + b x^2 +c x + d, and returns them in sorted order.
"""
function cubic_real_roots(b, c, d; verbose=false)::NTuple
    Δ = b^2 - 3*c
    μ = (2*(b^3)) - (9*b*c) + (27*d)

    if iszero(Δ) && iszero(μ) #Should check vs threshold.
        return (-b/3,) 
    end

    #Optimize: Add special statement for when Δ = 0 (within threshold.)
    L = (μ^2)-(4*Δ^3)
    mu_squared = (μ^2)
    four_delta = (4*Δ^3)
    if mu_squared < four_delta#L < 0 
        verbose ? println("L < 0") : nothing
        z = (μ + sqrt(-L)*im) #2z = ...,  but we only use the angle for z
        r2 = cbrt((μ^2 - L)/4)
        if r2 ≈ Δ #This must be handled more carefully!
            verbose ? println("r2 = Δ, difference: $(r2-Δ)") : nothing

            (s, c) = sincos(angle(z)/3) #can be optimized/altered to use the 'tan-formula' for the 3xReal root cubic
            k = 2*sqrt(r2)
            return sort((b .+ (k.* (c, (-c + (sqrt(3)*s))/2, (-c - (sqrt(3)*s))/2))) ./(-3))
        else
            verbose ? println("r2 ≠ Δ, difference: $(r2-Δ)") : nothing
            if μ ≤ 0
                verbose ? println("μ ≤ 0") : nothing
                return (b + sqrt(r2)*(1+(Δ/(r2))),) ./(-3)
            else
                verbose ? println("μ > 0") : nothing
                return (b - sqrt(r2)*(1+(Δ/(r2))),)./(-3)
            end
        end
    elseif mu_squared ≥ four_delta #L ≥ 0
        verbose ? println("L ≥ 0, L: $L") : nothing
        root = sqrt(L)
        if μ > 0
            z = (μ + root)/2
        else
            z = (μ - root)/2
        end
            
        C = cbrt(z)
        r2 = C^2
        if r2 ≈ Δ #This must be handled more carefully! We should check roots (amounts to a single computation) 
            verbose ? println("r2 = Δ, difference: $(r2-Δ)") : nothing
            if C < 0
                return (b+2*C, b-C) ./ (-3)
            end
            return (b-C, b+2*C) ./ (-3)
        else
            verbose ? println("r2 ≠ Δ, difference: $(r2-Δ)") : nothing
            return (b + C*(1+(Δ/r2)),) ./(-3)
        end
    end
end

cubic_real_roots

Thus we have established a pretty decent root-finder. Let's look at its performance:

In [222]:
bm_roots_1R = @benchmark cubic_real_roots($2.0, $(-3.0), $9.0) #One real root

BenchmarkTools.Trial: 10000 samples with 984 evaluations per sample.
 Range (min … max):  55.081 ns … 256.809 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     56.809 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   59.334 ns ±   8.024 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▆ █▆ ▇▂         ▂▁▂▂▁                                        ▁
  █▅█████▇██▆▇██████████▆▇▆▇▆▆▆▆▆▇▇▇█▇▆▆▆▆▆▇▆▆▆▆▆▆▆▆▇▅▅▅▅▄▄▅▅▅ █
  55.1 ns       Histogram: log(frequency) by time      90.1 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [223]:
bm_roots_3R = @benchmark cubic_real_roots($(-7.0), $(3.0), $9.0) #Three real roots

BenchmarkTools.Trial: 10000 samples with 951 evaluations per sample.
 Range (min … max):   92.008 ns … 354.364 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):      97.581 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   106.640 ns ±  20.181 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▄▄█▆▆▂ ▄▃▆▃▅▂ ▂▁▃▂▁    ▁                ▁▁▂▂▁ ▁               ▂
  ████████████████████▇████▇▇▇▆▆▆▅▆▆▅▅▆▆▇▇████████▇██▆▇▆▆▆▆▆▆▅▅ █
  92 ns         Histogram: log(frequency) by time        176 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

This is something like $\sim 10-15$ times faster than the one defined above. As a rough measure of performance then we expect something like $\sim 80 N ns$ to compute the partition of the time interval for a velocity update. For e.g. the 20-dimensional Twin Peaks scenario $N = 21$ so we get something akin to $\sim 1.6 \mu s$ of work. Of course, if we wanted to we could parallellize this particular task, but the overhead for such an endeavour would probably be prohibitive. 

We need to handle some special cases as well: When the coefficients are zero (possibly some analysis of tolerance would be useful here) we get somewhat simpler equations to handle.

In [224]:
function quadratic_real_roots(c, d)
    x1=c^2
    x2=4*d
    #Δ = c^2 - 4*d
    if x1<x2#Δ < 0
        return (Inf,)
    end
    if x1 ≈ x2 
        return (-c/2,)
    end
    s = sqrt(x1-x2)
    return ((-c-s)/2.0, (-c + s)/2.0 )
end

#If the polynomial is identically zero the rates never change, so we are fine to ignore such points.
#Julia doesn't love to handle combinations of empty and non-empty tuples.
#Thus we make the "no root exists" just assign Inf. This is goofy, but does make the compiler happy...
linear_real_roots(c, d) = iszero(c) ? (Inf,) : (-d/c,)

quadratic_real_roots(b,c,d) = iszero(b) ? linear_real_roots(c, d) : quadratic_real_roots(c/b, d/b)

cubic_real_roots(a,b,c,d) = (iszero(a) ? quadratic_real_roots(b,c,d) : cubic_real_roots(b/a, c/a, d/a))

cubic_real_roots(X::Vector) = cubic_real_roots(X...)

cubic_real_roots (generic function with 3 methods)

In [225]:
@benchmark cubic_real_roots($1.0, $2.0, $3.0, $3.0)

BenchmarkTools.Trial: 10000 samples with 980 evaluations per sample.
 Range (min … max):  61.122 ns … 222.857 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     64.184 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   67.065 ns ±   9.024 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▃▄▃██▆▆▆▃ ▁    ▁▁▂▂▃▃▁                                       ▂
  ███████████▇▇█▇████████▇▇▇▆▇▆▇▆█████▆▆▆▆▇▆▅██▇▇▇▇▇▇▅▅▅▆▄▆▅▄▅ █
  61.1 ns       Histogram: log(frequency) by time        99 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

#### Partitioning the velocity space
We know can fetch the zeros of the rho-functions, as they depend on $v^J = u$. To partition the time intervals we would have to determine which $u(t)$ is relevant of the above 6, and then invert that $u(t)$ as above, and compute $t(u_i)$ for each root. But since we are not interested in all roots, and since $u$ is - obviously - monotonic, it suffices that we partition the *velocity space* explicitly by ordering the roots that are greater or smaller than $u(0)=u_0$ and then apply the inverse only when we know we need to determine $t(u_i)$, which generally wont happen for all $u_i$. Thus the partitioning of the time space is not done explicitly, but rather when necessary.

What is notable though is that we are really after partitioning the velocity space according to the points at which the polynomials *flips their sign*. Moreover we want to know what sign a function has on either side of its roots, which can mostly, but not fully, be inferred from the sign of the leading non-zero coefficient. In edge cases, the functions do not even change their sign at roots. 

Let us list the possibilities, from 'most extreme' to least. For $p(x) = ax^3+bx^2+cx+d$ we have

1. The polynomial is constant, $a = b = c = 0$. No sign changes happen at all.

2. The polynomial is linear, $a = b = 0$. A single sign change happens att $r_1$ and if $c$ is positive it goes from negative to positive, and vice versa.

3. The polynomial is quadratic $a = 0$. If the polynomial has a single root it retains its sign throughout the whole time interval. If it has two roots the sign of the polynomial is determined by $a$
 
4. The polynomial is cubic. If it has a single root the signs can be fully inferred from $a$. If it has two roots then we can partially infer the sign from looking at the coefficient $a$ - to complement this we observe that one root is a double root $r_i$ (satisfying $p'(x)=3ax^2+2bx +c = 0$), and can be disregarded. The sign only flips at 'the other' root. If it has three roots the sign is fully inferrable from $a$.

Later we shall want to track a which polynomials are positive/negative and so we create a vector $\bar{S}= (\pm_1,\pm_2,\ldots, \pm_n)$ and when we encounter a root for which one of these flip a sign we encode not only what the root is, but also which one should flip. Of course, the sign is tracked relative to $u_0$ and in the direction $u(t)$ increases. 

In [226]:
#Removes roots of polynomials that do not correspond to a sign flip. Only applied to 2nd or 3rd order polynomials.
function filter_sign_flips(roots, pol_coeff)
    root_nr = length(roots)
    if iszero(pol_coeff[1]) && !iszero(pol_coeff[2])# degree 2
        if root_nr < 2
            return (Inf,) #we disregard a double root or no-roots
        else
            return roots
        end
    end
    if !iszero(pol_coeff[1])
        if root_nr == 2
            d1 = abs((3*pol_coeff[1]*(roots[1]^2)) +(2*pol_coeff[2]*roots[1]) + pol_coeff[3])
            d2 = abs((3*pol_coeff[1]*(roots[2]^2)) +(2*pol_coeff[2]*roots[2]) + pol_coeff[3])
            if d1 < d2
                if d1 ≈ 0 
                    return (roots[2],)
                else
                    @show d1, d2
                    error("Non-zero derivative double root in 3rd order polynomial")
                end
            elseif d2 < d1
                if d2 ≈ 0 
                    return (roots[1],)
                else
                    @show d1, d2
                    error("Non-zero derivative double root in 3rd order polynomial")
                end
            else 
                if !(d1 ≈ 0) 
                    @show d1, d2
                    error("Non-zero derivative double root in 3rd order polynomial")
                end
                if roots[1] ≈ roots[2]
                    return (roots[1],)
                end
                #At this point the roots are distinct but their derivative values are (within our precision) equal
                #THIS IS A FRINGE CASE, as it relies on several float-expressions being near 0.
                #We check the sign in between the roots to check which is proper 
                x = (roots[1] + roots[2])/2
                intermediate_value = dot(pol_coeff, (x^3, x^2, x, 1.0))
                if a > 0 
                    if intermediate_value >  0
                        return (roots[1],)
                    elseif intermediate_value  < 0
                        return (roots[2],)
                    else
                        #If the roots are 0 and the intermediate value is close to zero and the derivatives at the roots are near zero
                        #we expect that the whole polynomial is very close to 0 (coefficients may be withing a few machine ϵ from 0).
                        #We can either throw an error or we can accept that we may not be able to discern the polynomial from 0 
                        #Consequently the polynomial integrates - within our precision, to 0
                        return (roots[1],)
                    end
                elseif a<0
                    if intermediate_value >  0
                        return (roots[2],)
                    elseif intermediate_value  < 0
                        return (roots[1],)
                    else
                        return (roots[1],)

                    end
                else 
                    error("3 roots for a 2nd order polynomial?")
                end
            end
        end
    end
    return roots
end


filter_sign_flips (generic function with 1 method)

In [227]:
function partition_by_roots!(T::BinaryMinHeap, J::Integer, ρJ::Array{Float64, 2}, dir::Integer, u0::Float64) 
    empty!(T)
    if dir == 0 
        return T
    end

    for I in axes(ρJ, 1)
        if I ≠ J
            ρJI = @view(ρJ[I,:])
            roots = cubic_real_roots(ρJI[1], ρJI[2], ρJI[3], ρJI[4])
            if length(roots) > 1
                reduced_roots = filter_sign_flips(roots, ρJI)
            else
                reduced_roots = roots
            end
            add_roots = false
            if dir > 0 
                for i in eachindex(reduced_roots)
                    if !add_roots
                        if reduced_roots[i] > u0
                            add_roots = true
                            push!(T, (reduced_roots[i], I))
                        end
                    else
                        push!(T, (reduced_roots[i], I))
                    end
                end
            else
                for i in eachindex(reduced_roots)
                    if !add_roots
                        if reduced_roots[i] < u0
                            add_roots = true
                            push!(T, (reduced_roots[i], I))
                        end
                    else
                        push!(T, (reduced_roots[i], I))
                    end
                end
            end
        end
    end
    return T
end

partition_by_roots! (generic function with 1 method)

In [228]:
D = 21
T = BinaryMinHeap{Tuple{Float64, Int64}}()
ρ = randn(21, 4);
u0 = rand()
T = partition_by_roots!(T, 1, ρ, 1, u0)

BinaryMinHeap{Tuple{Float64, Int64}}(Base.Order.ForwardOrdering(), [(0.21302690334371235, 3), (0.3132190573365207, 8), (0.6485100181814839, 13), (0.4447349356414563, 2), (0.6350216047198808, 10), (1.9357678437986774, 6), (1.0258112145548937, 10), (2.0356676485266916, 14), (1.0635225083600393, 19), (1.6895812408069137, 21)])

In [229]:
@benchmark partition_by_roots!($T, $1, $ρ, $1, $u0)

BenchmarkTools.Trial: 10000 samples with 10 evaluations per sample.
 Range (min … max):  1.660 μs …  1.419 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.410 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   3.225 μs ± 17.939 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▄ ▆█▁▂                                                     ▁
  █▅████▇▅▅▆▆▆▆▇▅▄▆▅▆▆▅▅▅▅▅▄▄▄▅▄▁▄▄▃▄▄▄▁▄▁▃▁▃▄▁▁▃▃▁▄▃▁▄▄▁▁▄▄ █
  1.66 μs      Histogram: log(frequency) by time     15.6 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

We get roughly to $1.9\mu s$ of computation time.

### The rate integrals on a particular time-partition
#### A compact form for the rates

$\lambda = A u^3 + B u^2 + C u + D = \bar{A} \cdot \bar{U}$

over time (on the intervals discussed above). Here $\bar{A}^t = (A,B,C,D)$ and $\bar{U}^t = (U_3, U_2, U_1, U_0)$ (yes, this is annoying as e.g. $(\bar{U})_4 = U_0$ - yet it seems more unforunate to associate $u^3$ to $U_0$) where

$U_n = \int u(t)^n dt$

can be analytically computed, but the specific form depends crucially on which of the 6 forms above $u(t)$ takes. However, generally $u(t) = \beta(\alpha + g(t))$ so 

$u^n = \beta^n(\alpha +  g(t))^n = \beta^n \sum_{j=0}^n \binom{n}{j} \alpha^{n-j} g(t)^{j}$

so if $\int g^n = G_n$, and $M_{nj} = \beta^n \binom{n}{j}\alpha^{n-j}$ the integral form is

$U_n = \int u^n = \sum_{j=0}^n M_{nj} G_j$

and of course we can think of this as a matrix multiplication $\bar{U} = M\bar{G}$ for $\bar{G}^t = (G_3,G_2,G_1,G_0)$ if we set

$M_{ij} = 0$ if $i\leq j$ and otherwise $M_{ij} =  \beta^i \binom{i}{j}\alpha^{i-j}$.

This is very convenient as we shall evaluate the $G$ integrals in many different places, whereas the $M$-computation is the same over and over for each dynamic. Of course, for some velocity types there may be close relationships between the integrals of e.g. $g(t)^2$ and, say, $1$.  For those it might make sense to store even more information between evaluations to avoid unnecessary calculations.

As the time intervals change we may get new $A,B,C,D$ (remember - these are essentially our $[\rho_{JI}]^+$, which we ignore when the expression $\lambda$ is non-positive. Thus they really only come into play when $\lambda$ flips its sign) but $M$ only depends on the dynamic $\text{dyn}_J$, and $U_n$ truly only depends on the integration time interval and the dynamic. Thus, for any $I$ we can summarize 

$\Lambda_{JI} = \int \lambda_{JI} dt = \bar{[\rho_{JI}]^+}^t M_{\text{dyn}_J} \bar{G}(T)$

and thus we can arrange the $\Lambda_{JI}$ into a matrix $\Lambda^i_{J} = \Lambda_{Ji_1}| \Lambda_{Ji_2} | \ldots|\Lambda_{Ji_m}$ where $m$ is however many non-zero rates we are dealing with at the moment. This is not of quite the same use - the form of the matrix will vary, and while we could compute it may not offer much convenience.

In [230]:
function M_matrix!(M::MMatrix{4,4, Float64, 16}, α, β)
    S = @SMatrix [1.0 2.0 3.0 4.0; 0.0 1.0 3.0 6.0; 0.0 0.0 1.0 4.0; 0.0 0.0 0.0 1.0]
    for i in 1:4
        for j in i:4
            M[i, j] = (α^(i-j))
        end
        factor = (β^i)
        @views M[i,:] .*= factor
    end
    M .*= S
    return M
end

M_matrix! (generic function with 1 method)

#### Looking forward: Efficient computation of forward and reverse rates.
Because of our careful management of the partition above we can figure out the rates at the beginning of an evolution, for each outgoing state $I$ or $n+1$, and track sign changes relative to that original configuration. Remember that we want to compute the forward rate integral $\Lambda$, and terminate when $\Lambda = C$ for some $C > 0$. We also want to compute $\Lambda^{rev}$. Both are, on a set $T_i$ in our partition, entirely determined by the integrals of our $\rho_{JI}$ and $\rho_{J,n+1}$. Specifically, $\Lambda_{JI,i} = \int_{T_i} \sum_{u \in P_i} \rho_{Ju}dt$ where $P_i$ enumerates the set of $\rho_{Ju}$ that are positive in $T_i$. The reversed lambda integral $\Lambda^{rev}$ is the sum of the (absolute value) of the corresponding expression summed over $u \notin P_i$.

We can can implement the following algorithm to compute $\Lambda, \Lambda^{rev}$ etc.

1. Begin at velocity $v_0$ and $t= t_{prev}= 0$ and check the values of $\lambda_{Ju}$ for each $u$ (including $n+1$). Create a vector $\bar{S}$ of length $n$ containing the signs of each $\rho_{Iu}$ at $v_0$.

2. Let $P$ denote the $u$ for which the rate is positive, and $N$ its complement. Create vectors 

$\bar{F}_P = (\sum_{u \in P} \bar{\rho}_{JI})^t M_{dyn}$ and $\bar{F}_N = (\sum_{u \in N} \bar{\rho}_{JI})^t M_{dyn}$

3. Create a binary heap $B$ of all doubles $(v, u)$, listing sign flips that occur after $v_0$, ordered so that the first sign flip of a $\rho_{JI}$ comes first. Associated to each sign flip is a $u$ pointing to which $\rho_{Ju}$ it is that changes its sign.

4. Now loop until $\Lambda \geq C$ - actually this will always terminate in step e) below so could be "while true"
    
   a) Let $(v, u)$ be the first element in the binary heap. Remove it from the heap

   b) Let $t_{prev} = t$. Invert $v$ and set $t = t_{dyn}(v)$ using the inverse that depends on the dynamic.

   c) Compute $\Delta \bar{G} = \bar{G}(t) - \bar{G}(t_{prev})$ and compute $\Delta \Lambda = \Delta \bar{G}\cdot \bar{F}_P$ 

   d) If $\Delta \Lambda < C - \Lambda$ set $\Lambda = \Lambda + \Delta \Lambda$, and set $\Lambda^{rev} = \Lambda^{rev} + \Delta\bar{G}\cdot \bar{F}_N $
    - Check the sign of the $u$:th element of the sign vector to see which element flips from positive to negative. Flip $S[u] = -S[u]$
    - Adjust the $\bar{F}_N$ and $\bar{F}_P$ to adjust the $u$ contribution to the $\bar{F}$ vectors (adding and subtracting some $\rho_{Ju}^tM_{dyn}$ from both).

   e) If the sum exceeds or equals $C-\Lambda$ we should terminate in some point possibly before $v$. 
    - Compute $t$ such that $\bar{G}(t)\cdot \bar{F}_P = C - \Lambda + \bar{G}(t_{prev})\cdot \bar{F}_P$ using bisection or some such. 
    - Set $\Lambda = \Lambda + \Delta \Lambda$, and set $\Lambda^{rev} = \Lambda^{rev} + \Delta\bar{G}\cdot \bar{F}_N $
    - Break the loop.

5. Return $\Lambda, \Lambda^{rev}, t, u(t)$.

Here we see a computational feature of these rates. We never use $\rho$, but always $\rho M$ so we should not store $\rho$ generally - rather we should store $\rho M$.
There is some further optimizations in handling cancellations between integral terms, but this gives us a good picture of how the final algorithm will look.

#### Rate integral for type 0 velocity
This is one trivial case for which we do not have much use of any advanced forms. We get

$\Rho_{JI} = \int \bar{A} \cdot \bar{U}_0 dt = \Delta T \rho_{JI}$

For this class of dynamics we do not use the machinery described above. We shall not go into detail here.

#### Rate integral for type 1 velocity
We have $u(t) = ct +u_0$ so we have to integrate

$\int (ct + u_0)^ndt$

whence we get $M$-matrices $M(\alpha, \beta)$ with $ \beta = c$ and $\alpha = u_0/c$ and get for $j\geq 1$

$G_j = \int t^j dt = \frac{1}{j+1}T^{j+1}$

In [231]:
function G_type_1(T, c, u0, inv_c)
    return (T^4, T^3, T^2, 1.0) ./ (4.,3.,2.,1.)
end

G_type_1 (generic function with 1 method)

#### Rate integral for type 2 velocity
We have $u(t) = ct +u_0$ so we have to integrate

$\int (me^{bt} - q)^ndt$

whence we get $M$-matrices $M(\alpha, \beta)$ with $ \beta = m$ and $\alpha = -q/m$ and get for $j\geq 1$

$G_j = \int e^{btj} dt = \frac{1}{jb}e^{jbT}$

In [232]:
#Param: b, q, m
function G_type_2(T, b, q, m)
    s = exp(b*T)
    return ((s^3)/3.0,(s^2)/2.0,s,T) ./ b
end

G_type_2 (generic function with 1 method)

#### Rate integral for type 3 velocity
We have $u(t) = \frac{1}{a}(\frac{1}{l_0-t} - \delta)$ so we have to integrate

$\int (\frac{1}{a}(\frac{1}{l_0-t} - \delta))^ndt$ 

whence we get $M$-matrices $M(\alpha, \beta)$ with $ \beta = a^{-1}$ and $\alpha = -\delta$ and get for $j\geq 2$

$G_j = \int \frac{1}{(l_0-t)^j} dt = \frac{1}{j-1}(l_0 - T)^{1-j}$

and for $j = 1$

$G_1 = -\log (l_0-T)$

In [ ]:
#Param: (a_inv, l0_inv, l0, δ, a) 
function G_type_3(T, a_inv, l0_inv, l0, δ, a)
    l0 = l0
    Δ = l0 - T
    Δ_inv = inv(Δ)
    return ((Δ_inv^2)/2.0, Δ_inv,-log(Δ),T) 
end

G_type_3 (generic function with 1 method)

#### Rate integral for type 4 velocity
We have $u(t) = \frac{1}{a}((k \tan(s_0 + kt)) - δ)$ so we have to integrate

$\int (\frac{1}{a}((k \tan(s_0 + kt)) - δ))^ndt$ 

whence we get $M$-matrices $M(\alpha, \beta)$ with $ \beta =  ka^{-1}$ and $\alpha = -\delta/k$ and get for $j =  1$

$G_1 = \int \tan(s_0 + kt) dt = -\frac{1}{2k}\log(\cos^2(s_0 + kt))$

and for $j=2$

$G_2 = \int \frac{(2k)^2}{(1-\omega_0 \exp(2kt))^2} dt = \frac{1}{k}(\tan(s_0+kt) -  s_0 - kt)$

and for $j=3$

$G_3 = \int \tan(s_0 + kt)^3 dt = \frac{1}{2k}(\tan^2(s_0+kT) + \log(\cos^2(s_0+kt)))$


In [234]:
#Parameters (a_inv, k, s0, δ, k_inv, a)
function G_type_4(T, a_inv, k, s0, δ, k_inv, a)
    X = (k*T) + s0
    sX, cX  = sincos(X)
    lcX = log(cX^2)
    tX = sX/cX
    return ( tX^2 + lcX, tX-X, lcX, T) .* (k_inv/2.0, k_inv, -k_inv/2.0, 1.0)
end

G_type_4 (generic function with 1 method)

#### Rate integral for type 5 velocity
We have $u(t) = \frac{1}{a}(-l + \frac{2k}{1 - \omega_0exp(2kt)})$ so we have to integrate

$\int (\frac{1}{a}(-l + \frac{2k}{1 - \omega_0exp(2kt)}))^ndt$ 

whence we get $M$-matrices $M(\alpha, \beta)$ with $ \beta =  a^{-1}$ and $\alpha = -l$ and get for $j =  1$

$G_1 = \int \frac{2k}{(1-\omega_0 \exp(2kt))} dt = 2k T - \log(|k\omega_0 \exp(2kT) -k|)$

and for $j=2$


$G_2 = \int \frac{(2k)^2}{(1-\omega_0 \exp(2kt))^2} dt = 2k(\frac{1}{1-\omega_0 \exp(2kT)} + 2k T - \log(|k\omega_0 \exp(2kT) -k|))$

and for $j=3$

$G_3 = \int \frac{(2k)^3}{(1-\omega_0 \exp(2kt))^3} dt = -4k^2(\frac{\omega_0 e^{2kT} -3}{(\omega_0 \exp(2kT) -1)^2} -  (2kT - \log(|k\omega_0 \exp(2kT) -k|)))$


In [235]:
#Parameters (a_inv, δk, dk, ω0, a)
function G_type_5(T, a_inv, δk, dk, ω0, a)
    X = dk*T 
    Y = ω0*exp(X)-1.
    Z = abs(dk*Y/2.0)
    first_term = X-log(Z)
    exp_inverse= inv(Y)
    second_term = dk * ( first_term - exp_inverse)
    third_term = (dk^2) * (first_term - ((Y - 0.5) *(exp_inverse^2)))
    return (third_term, second_term, first_term, T)
end

G_type_5 (generic function with 1 method)

### Implementing the rate integration algorithm
Combining almost all of the above we get to our flow in the $J$:th velocity dynamic.

In [236]:
struct FixedParameterIntegration{T,I}
    parameters::T
    integral::I
end

(G::FixedParameterIntegration)(t) = G.integral(t, G.parameters...)# evaluate(G.V, t, G.parameters)#evaluate_integral(G.V, t, G.parameters)

In [237]:
(dir, param, flow_type) = direction_velocity_and_type(0.3, 0.2, 1.3, -0.2)
G = FixedParameterIntegration(param, flow_type.integral)

FixedParameterIntegration{NTuple{6, Float64}, typeof(G_type_4)}((3.3333333333333335, 1.2328828005937953, 0.032432907451242506, 0.1, 0.8111071056538127, 0.3), Main.G_type_4)

In [238]:
@code_warntype G.integral(0.2, G.parameters...)

MethodInstance for G_type_4(::Float64, ::Float64, ::Float64, ::Float64, ::Float64, ::Float64, ::Float64)
  from G_type_4(T, a_inv, k, s0, δ, k_inv, a) @ Main e:\Arbete\Lagrangian-PDMPs\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y156sZmlsZQ==.jl:2
Arguments
  #self#::Core.Const(Main.G_type_4)
  T::Float64
  a_inv::Float64
  k::Float64
  s0::Float64
  δ::Float64
  k_inv::Float64
  a::Float64
Locals
  @_9::Int64
  tX::Float64
  lcX::Float64
  cX::Float64
  sX::Float64
  X::Float64
Body::NTuple{4, Float64}
1 ─ %1  = Main.:+::Core.Const(+)
│   %2  = Main.:*::Core.Const(*)
│   %3  = (%2)(k, T)::Float64
│         (X = (%1)(%3, s0))
│   %5  = Main.sincos::Core.Const(sincos)
│   %6  = X::Float64
│   %7  = (%5)(%6)::Tuple{Float64, Float64}
│   %8  = Base.indexed_iterate(%7, 1)::Core.PartialStruct(Tuple{Float64, Int64}, Any[Float64, Core.Const(2)])
│         (sX = Core.getfield(%8, 1))
│         (@_9 = Core.getfield(%8, 2))
│   %11 = @_9::Core.Const(2)
│   %12 = Base.indexed_iterate(%7, 2,

In [239]:
function do_the_thing()
    compute_divergences!(A, reduced_v, dim, J,  Γ, G, G_inv, ∇φ, vel) #all terms here are to be collected into EvoData, except dim
    compute_rho!(ρJ, J, A, dim) #all terms here are to be collected into EvoData, except dim
    (a,b,c) = compute_velocity_parameters!(reduced_v, Γ, Ginv, ∇φ, J, v) #all terms here are to be collected into EvoData
    (dir, param, flow_type) = direction_velocity_and_type(a, b, c, u0)
    M_matrix = M_matrix!(M, α, β)
    mul!(M_adj_ρJ, M_matrix', ρJ)

    t = 0.0
    t_prev = 0.0
    
    ini_u_vector = (u0^3, u0^2, u0, 1.0) #EvoData
    
    λ0_vector = [dot(@view(M_adj_ρJ[I, :]), ini_u_vector) for I in 1:(dim+1)] #EvoData

    sign_vector = sgn.(λ0_vector) #EvoData

    F_positive = @MVector zeros(Float64, 4) #EvoData
    F_negative = @MVector zeros(Float64, 4) #EvoData

    G = FixedParameterIntegration(param, V)
    
    for I in eachindex(sign_vector)
        if iszero(sign_vector[I]) 
            if !(I == J)
                #Implement a function that checks the behaviour of ρ_{JI}. For now we throw an error
                error("Unimplemented! But should be implemented")
            end
        elseif sign_vector[I] > 0
            F_positive .+= M_adj_rho[J,:]
        else 
            F_negative .+= M_adj_rho[J,:]
        end
    end

    T = BinaryMinHeap{Tuple{Float64, Int64}}() #EvoData
    partition_by_roots!(T, J, ρJ, dir, u0) 
    Λ = 0.0
    Λ_rev = 0.0
    G = evaluate_integral(V, param, t)
    while Λ < threshold
        if isempty(T) 
            #Check for termination point
            Z(t) = (G(t) - G(t_prev) - threshold + Λ)
            t_new = find_zero(Z, (t_prev, t), root_method) 
        else
            (u, I) = pop!(T)
        end



        t_prev = t
        t = evaluate_inverse(V, param, u)

        G_prev= G(t_prev)      
        ΔG = G(t) - G_prev
        
        ΔΛ = dot(F_positive, ΔG)
        if Λ + ΔΛ ≥ threshold
            Z(t) = (G(t) - G_prev - threshold + Λ)
            t_new = find_zero(Z, (t_prev, t), root_method) 
            
            break
        else
            Λ += ΔΛ
            Λ_rev -= dot(F_negative, ΔG)
            if sign_vector[I] > 0
                F_positive .-= @view(M_adj_ρ[I,:])
                F_negative .+= @view(M_adj_ρ[I,:])
            else
                F_positive .+= @view(M_adj_ρ[I,:])
                F_positive .-= @view(M_adj_ρ[I,:])
            end            
            sign_vector[I] = -sign_vector[I]
        end
        

Base.Meta.ParseError: ParseError:
# Error @ e:\Arbete\Lagrangian-PDMPs\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y163sZmlsZQ==.jl:76:9
        end
        
#       └ ── Expected `end`